# Converging nozzle — solving a flow network from a UI-exported YAML case

This notebook loads a compressible-flow network **saved from the FNetLibUI tool**
(the `fns-flow-network` model), solves for the steady mean flow with **FNS**, and
then runs a linear **acoustic transfer / scattering** analysis on top of the
converged mean flow.

The case is a *converging nozzle* with pipework:

    reservoir --(feed)--> Duct --(pipe)--> IsentropicAreaChange
              --(throat)--> Duct --(tailpipe)--> back-pressure

A high-pressure reservoir discharges through a feed pipe, a smooth area
contraction, and a tailpipe to a fixed back pressure. The two ducts are
constant-area: **inert in the mean flow** (duct length never enters the steady
residual) but they **carry the acoustic phase** used by the perturbation network.

The YAML is the UI's native export. Edges reference **ports** explicitly through
their `sourceHandle` / `targetHandle` (`<node>-port-<ordinal>`); the loader
preserves those ordinals.

All figures use the project's shared plotly theme, `fns.plotting`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # make the `fns` package importable

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from fns.io import load_case
from fns.plotting import use_fns_theme, COLORWAY

use_fns_theme()   # register + activate the modern FNS plotly template

CASE = "converging_nozzle.yaml"
print(open(CASE).read())

# Converging nozzle -- saved from the FNetLibUI tool (model: fns-flow-network).
#
#   reservoir --(feed)--> Duct --(pipe)--> IsentropicAreaChange
#             --(throat)--> Duct --(tailpipe)--> back-pressure
#
# A high-pressure reservoir discharges through a feed pipe, a smooth area
# contraction, and a tailpipe to a fixed back pressure.  The two ducts are
# constant-area: they are inert in the mean flow (duct length does not enter the
# steady residual) but carry the acoustic phase used by the perturbation network.
#
# This is the UI tool's native export format.  Note that edges reference PORTS
# explicitly via sourceHandle / targetHandle (`<node>-port-<ordinal>`): port 0 is
# the target (incoming) side of an element, port 1 the source (outgoing) side.

version: 2.0.0
model:
  id: fns-flow-network
  globalAttributes:
    gasConstant: 287.0
    heatCapacityRatio: 1.4
    referencePressure: 101325.0
    referenceTemperature: 300.0
    referenceMassFlow: 5.0
  nodes:
    - id: TotalPres

## 1. Load the UI case and solve

`load_case` parses `model.globalAttributes` (gas + reference scales),
`model.nodes` (element `type` + `attributes`) and `model.edges` (with their port
handles and area) into a `Network`. `Network.solve()` compiles the immutable
problem and runs the damped Newton solve from a quiescent start.

In [2]:
net = load_case(CASE)
sol = net.solve()

print("converged :", sol.converged)
print("iterations:", sol.iterations)
print("||R_hat|| :", f"{sol.residual_norm:.2e}")

converged : True
iterations: 8
||R_hat|| : 3.05e-12


## 2. Inspect the converged edge states

Every edge carries the recovered state `(mdot, p, h_t, rho, u, T, c, M, p_t)`.
Total pressure is uniform across the lossless contraction; the throat and
tailpipe (area `0.010`) run faster than the feed pipe (area `0.020`).

In [3]:
prob = net.compile()
names = net._edge_names
fields = ["mdot", "M", "p", "p_t", "T", "rho"]

hdr = f"{'edge':>10} " + " ".join(f"{f:>10}" for f in fields)
print(hdr); print("-" * len(hdr))
for e in range(prob.n_edges):
    s = sol.edge(e)
    row = " ".join(f"{s[f]:>10.4g}" for f in fields)
    print(f"{names[e]:>10} " + row)

THROAT = names.index("throat")   # the choking edge

      edge       mdot          M          p        p_t          T        rho
----------------------------------------------------------------------------
      feed      4.125     0.2668  1.903e+05      2e+05      295.8      2.242
      pipe      4.125     0.2668  1.903e+05      2e+05      295.8      2.242
    throat      4.125     0.6545    1.5e+05      2e+05      276.3      1.891
  tailpipe      4.125     0.6545    1.5e+05      2e+05      276.3      1.891


## 3. Back-pressure sweep: emergent choking

Lower the outlet static pressure and re-solve. Above the critical ratio
($p/p_t \approx 0.528$ for air) the flow is subsonic and the mass flow rises as the
back pressure drops. Below it the **throat chokes**: it reaches $M=1$ and the mass
flow saturates at the analytic maximum

$$\dot m_{\max} = \frac{p_t}{\sqrt{T_t}}\,A^\ast\,\sqrt{\tfrac{\gamma}{R}}\left(\tfrac{2}{\gamma+1}\right)^{\frac{\gamma+1}{2(\gamma-1)}}.$$

No regime switch is coded — choking emerges from the smooth complementarity row.
(We re-solve the loaded network while overriding the outlet's back-pressure
parameter.)

In [4]:
pt, Tt, R, gamma = 200000.0, 300.0, 287.0, 1.4
A_throat = net._edges[THROAT][2]
flux_star = np.sqrt(gamma / R) * (2.0 / (gamma + 1.0)) ** ((gamma + 1.0) / (2.0 * (gamma - 1.0)))
mdot_max = pt / np.sqrt(Tt) * flux_star * A_throat
p_crit = (2.0 / (gamma + 1.0)) ** (gamma / (gamma - 1.0)) * pt

ratios = np.linspace(0.30, 0.95, 26)
mdots, machs = [], []
for r in ratios:
    n = load_case(CASE)
    n._elements[-1].fparams[0] = r * pt   # override the back-pressure outlet's p
    s = n.solve()
    est = s.edge(THROAT)
    mdots.append(est["mdot"]); machs.append(est["M"])

mdots, machs = np.array(mdots), np.array(machs)
print(f"throat area         = {A_throat:.4f} m^2")
print(f"mdot_max (analytic) = {mdot_max:.4f} kg/s")
print(f"critical ratio      = {p_crit/pt:.4f}")

throat area         = 0.0100 m^2
mdot_max (analytic) = 4.6671 kg/s
critical ratio      = 0.5283


In [5]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Mass-flow saturation (choking)", "Throat reaches M = 1"),
    horizontal_spacing=0.12,
)

fig.add_trace(go.Scatter(x=ratios, y=mdots, mode="lines+markers", name="solved"), row=1, col=1)
fig.add_hline(y=mdot_max, line=dict(dash="dash", color=COLORWAY[4]),
              annotation_text="ṁ_max", annotation_position="bottom right", row=1, col=1)
fig.add_vline(x=p_crit / pt, line=dict(dash="dot", color="#52606d"),
              annotation_text="critical ratio", annotation_position="top left", row=1, col=1)

fig.add_trace(go.Scatter(x=ratios, y=machs, mode="lines+markers", name="throat M",
                         line=dict(color=COLORWAY[2])), row=1, col=2)
fig.add_hline(y=1.0, line=dict(dash="dash", color=COLORWAY[4]), row=1, col=2)
fig.add_vline(x=p_crit / pt, line=dict(dash="dot", color="#52606d"), row=1, col=2)

fig.update_xaxes(title_text="back-pressure ratio  p_out / p_t", row=1, col=1)
fig.update_xaxes(title_text="back-pressure ratio  p_out / p_t", row=1, col=2)
fig.update_yaxes(title_text="throat mass flow  ṁ  [kg/s]", row=1, col=1)
fig.update_yaxes(title_text="throat Mach number", row=1, col=2)
fig.update_layout(height=440, width=960, showlegend=False, hovermode="x",
                  title_text="Emergent choking of the converging nozzle")
fig.show()

## 4. Perturbation network: transfer & scattering matrices

The same compiled network supports a linear **perturbation** analysis on top of
the mean flow (theory §12). The 1-D Euler system carries **three** characteristics
— two acoustic ($f$, $g$) and one **entropy / convected** wave ($h$) — so the
perturbation network has the *same variable count as the mean flow*, and with the
entropy wave its transfer / scattering matrices are genuinely **3×3** (growing
further with reacting scalars).

`perturbation_response` freezes the converged mean state, builds the perturbation
operator $A(\omega)$, and forces it **once** per frequency. **Every** incoming wave
is *prescribed* — the two acoustic terminals on their boundary rows, the incoming
entropy wave on the inlet edge's total-enthalpy transport row (the edge view of
nodal energy conservation) — so nothing is ever left floating at a boundary. The
`excite` argument chooses which families are *driven*: the default `("acoustic",)`
drives the acoustic waves and pins the incoming entropy to **zero** (a clean,
well-conditioned **2×2** acoustic response); adding `"entropy"` drives the entropy
wave too for the full **3×3**. Any transfer matrix between an edge pair, or the
scattering matrix between the terminals, is then reconstructed for the whole
frequency array **without re-solving**.

We drive the full set, read the $3\times3$ transfer matrix $T(\omega)$ (in the
characteristic flavor $f,g,h$) and the $3\times3$ scattering matrix $S(\omega)$,
plus the clean acoustic **2×2** — then plot them with the `fns.plotting`
complex-matrix viewers (magnitude over phase, one cell per entry).

In [6]:
from fns.perturbation import perturbation_response, find_terminals, verify_perturbation

verify_perturbation(prob, sol.x)   # subsonic, flow-aligned, length > 0 — preconditions OK

terms = find_terminals(prob, sol.x)
inlet = next(t for t in terms if t.at_tail).edge       # 'feed'     (f enters; carries entropy in)
outlet = next(t for t in terms if not t.at_tail).edge  # 'tailpipe' (g enters)
print(f"inlet  edge = {names[inlet]} (e{inlet})")
print(f"outlet edge = {names[outlet]} (e{outlet})")

freq = np.linspace(20.0, 1500.0, 600)            # Hz
# Drive the entropy wave too for the full 3x3.  The incoming entropy is pinned to
# zero on the acoustic columns, so the acoustic sub-block stays clean either way.
resp = perturbation_response(prob, sol.x, 2 * np.pi * freq, excite=("acoustic", "entropy"))

T = resp.transfer_matrix(inlet, outlet)          # (n_freq, 3, 3) in (f, g, h)
S = resp.scattering_matrix(inlet, outlet)        # (n_freq, 3, 3) incoming -> outgoing
incoming, outgoing = resp.scattering_labels(inlet, outlet)
print("T, S shapes :", T.shape, S.shape)
print("S incoming  :", incoming, "\nS outgoing  :", outgoing)

# read off the familiar acoustic coefficients AND the entropy-noise coupling
R_in = S[:, 0, 0]           # reflection   f_in -> g_in       (incoming order: f_a, h_a, g_b)
T_fwd = S[:, 1, 0]          # transmission f_in -> f_out
entropy_sound = S[:, 1, 1]  # entropy noise: incoming entropy h_in -> downstream sound f_out
print(f"|T_fwd|         : {np.abs(T_fwd).min():.3f} .. {np.abs(T_fwd).max():.3f}")
print(f"|entropy->f_out|: {np.abs(entropy_sound).min():.3f} .. {np.abs(entropy_sound).max():.3f}")

inlet  edge = feed (e0)
outlet edge = tailpipe (e3)
T, S shapes : (600, 3, 3) (600, 3, 3)
S incoming  : [('a', 0), ('a', 2), ('b', 1)] 
S outgoing  : [('a', 1), ('b', 0), ('b', 2)]
|T_fwd|         : 1.103 .. 1.103
|entropy->f_out|: 11.010 .. 11.010


In [7]:
from fns.plotting import plot_transfer_matrix, plot_complex_matrix

# (a) the full 3x3 transfer matrix — two acoustic waves AND the entropy wave.
# Grid layout: one (magnitude-over-phase) cell per entry, labelled f→f, f→g, …
fig = plot_transfer_matrix(
    T, freq, basis="char", x_title="frequency  [Hz]", width=920, height=720,
    title=f"Perturbation transfer matrix  {names[inlet]} → {names[outlet]}  "
          f"(mean M_throat ≈ {sol.edge(THROAT)['M']:.2f})",
)
fig.show()

# (b) the clean acoustic 2x2 scattering matrix (incoming entropy pinned to zero) —
# flat preset (magnitude row over phase row, the familiar Bode array). Entries
# 11,12,21,22 = R_in, T_bwd, T_fwd, R_out. Magnitudes are flat (lossless ducts are
# pure delays); only the phases wind down with frequency.
S2 = resp.acoustic_scattering_matrix(inlet, outlet)   # (n_freq, 2, 2)
fig2 = plot_complex_matrix(
    S2, freq, layout="flat", x_title="frequency  [Hz]", width=1000, height=420,
    title="Acoustic scattering matrix  S(ω)  (rows: g_in, f_out;  cols: f_in, g_out)",
)
fig2.show()

The transfer-matrix **magnitudes are flat** in frequency: between these two
stations the only frequency dependence is the duct propagation, and in wave
variables a lossless duct is a *pure delay* — a unit-magnitude phase — so it winds
each entry's phase down linearly (the sawtooth ramps) without ever changing its
amplitude. The **entropy column** ($\,\cdot \leftarrow h$) is large: an incoming
temperature/density spot converts strongly to sound as it accelerates through the
contraction — the *entropy noise* an acoustics-only (2×2) model cannot see — and
its phase winds at the slow **convective** delay $L/u$, not the acoustic
$L/(c\pm u)$. The acoustic **2×2** block is the clean acoustic scattering matrix
with the incoming entropy pinned to zero, so it is flat in magnitude; nothing is
left floating at the boundary to fold the (large) entropy→sound coupling back into
the acoustic waves. The whole picture comes from a **single** forced solve per
frequency, re-used to extract any edge pair, all on the same converged mean flow
and Jacobian from §1.